In [1]:
# Cập nhật danh sách gói (bắt buộc)
!sudo apt-get update

# Cài đặt công cụ zstd để giải nén file .tar.zst của Ollama phiên bản mới
!sudo apt-get install -y zstd

# Cài đặt các thư viện Python
%pip install fastapi uvicorn ollama

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,990 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Hit:12 https://ppa.launchpadcontent.net/graphic

In [2]:
# Chạy script cài đặt Ollama (lúc này đã có zstd để giải nén)
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [25]:
import subprocess
import time

# (Tùy chọn) Kill tiến trình ollama cũ nếu có, để tránh xung đột cổng
subprocess.run("pkill ollama", shell=True, stderr=subprocess.DEVNULL)
time.sleep(1)

# Khởi động ollama serve trong background
# stdout và stderr được chuyển hướng để tránh làm lộn xộn output của notebook
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Đợi vài giây cho server khởi động hoàn toàn
time.sleep(8)
print("✅ Ollama server đã được khởi động thành công!")

✅ Ollama server đã được khởi động thành công!


In [26]:
# Kiểm tra xem server đã sẵn sàng chưa
!ollama list

# Tải model Qwen3:8b về (chỉ cần làm một lần)
!ollama pull qwen3:8b

]11;?\NAME        ID              SIZE      MODIFIED       
qwen3:8b    500a1f067a9f    5.2 GB    19 minutes ago    
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling a3de86cd1c13: 100% ▕██████████████████▏ 5.2 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 05a61d37b084: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
# Cell 5: Hàm gọi model (thay thế cho API)
from ollama import generate
from pydantic import BaseModel
import json
import re
from pathlib import Path

class Response(BaseModel):
    qtype: str


def load_prompt(filename: str, **kwargs) -> str:
    prompt_path = Path("/kaggle/input/datasets/nam1706/prompt-v1-2/"+ filename)
    template = prompt_path.read_text(encoding="utf-8")
    # Nếu có placeholder, thay thế bằng kwargs
    for key, value in kwargs.items():
        placeholder = "{{" + key + "}}"
        template = template.replace(placeholder, value)
    return template


def classify_with_llm(question: str) :
    prompt_text = load_prompt("prompt.txt", QUESTION="QUESTION: " + question)
    response = generate(
        model='qwen3:8b',
        prompt=prompt_text,
        options={
            'temperature': 0,
            'think':False,
            'stream':False,
            'format': Response.model_json_schema(),
            'keep_alive':'5m',
            'num_predict': 1000  # CHẶN: Không cho sinh quá 100 token
        }
    )
    # ... gọi ollama với prompt trên, temperature=0
    raw = response # giả sử đây là string trả về
    # LLM thường trả về markdown json```json\n{...}\n```, cần làm sạch
    # Dùng regex tìm khối JSON đầu tiên
    return raw['response']

In [80]:

q0= "Based on the premises, what can we conclude about the curriculum?\nA. It enhances student engagement but not critical thinking\nB. It enhances critical thinking\nC. It needs more resources to enhance critical thinking\nD. It is well-structured but lacks exercises"+"\nDoes the combination of faculty priorities and curriculum features lead to enhanced critical thinking?"

q1 = "Two electric charges q1 = +9×10⁻⁶ C and q2 = -9×10⁻⁶ C are placed 10 cm apart. A charge q3 = +9×10⁻⁶ C is at the midpoint. Bring the problem to SMT formula to find forces between q1 and q3"

q2 = "Calculate the energy stored in capacitor C when C = 100 μF and U = 30 V."
print(classify_with_llm(q1))


TypeError: string indices must be integers, not 'str'

In [66]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files[:5])

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/nam1706 []
/kaggle/input/datasets/nam1706/prompt-ver1-0 ['prompt.txt']
/kaggle/input/datasets/nam1706/prompt ['prompt.txt']
/kaggle/input/datasets/nam1706/prompt-v1-2 ['prompt.txt']
/kaggle/input/datasets/nam1706/prompt-v1-1 ['prompt.txt']
